# Day 9 — Top-k retrieval  ·  Milestone 1 capstone 🏁

**Milestone 1 finale:** turn similarity scores into the actual **retrieved documents**. By the end you'll have a tiny RAG retriever — and you'll *understand every line*.

## 1. Review (~5 min)

Recall **day 2** (build a matrix) and **day 6** (normalize).

In [ ]:
import numpy as np
rng = np.random.default_rng(9)

**R1.** (day 2) stack vectors into rows of an `(n, d)` matrix.

In [ ]:
def r_build_matrix(vectors):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r1(fn):
    vs = [rng.normal(size=3) for _ in range(4)]
    M = np.asarray(fn(vs))
    assert M.shape == (4, 3) and np.allclose(M, np.array(vs))
    print("✅ Review R1 passed")

check_r1(r_build_matrix)

**R2.** (day 6) normalize a vector to unit length.

In [ ]:
def r_normalize(x):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r2(fn):
    for _ in range(4):
        x = rng.normal(size=int(rng.integers(2, 8)))
        assert np.allclose(np.linalg.norm(np.asarray(fn(x))), 1.0)
    print("✅ Review R2 passed")

check_r2(r_normalize)

## 2. Lecture note (~10 min): from scores to results

**The itch.** Day 8 gave us a score per document. The user doesn't want scores — they want the **best few documents**.

**The picture.** Sort the documents by score, highest first, and take the top $k$. We only need the *indices*, so we can fetch those documents.

**The tools.**
- `np.argmax(s)` → the index of the single largest score (top-1).
- `np.argsort(s)` → indices that would sort `s` **ascending**; for top-$k$ **descending** use `np.argsort(-s)[:k]`.

So the full retriever is just: *embed query → cosine-score the corpus (day 8) → take the top-$k$ indices.*

**Knobs & effect.** $k$ trades recall for noise — more passages = more context but more distraction. (`np.argpartition` finds the top-$k$ faster without fully sorting, when $n$ is huge.)

**In the wild.** This is literally the **retrieve** step of RAG: pick the top-$k$ passages, then hand them to the LLM as context. You just built the 'R'.

## 3. Exercises (~15 min)

### Exercise 1 — top-1
Return the index of the largest entry of a 1-D score array `s`.

In [ ]:
def top1(s):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e1(fn):
    for _ in range(5):
        s = rng.normal(size=int(rng.integers(2, 10)))
        assert fn(s) == int(np.argmax(s))
    print("✅ Exercise 1 passed")

check_e1(top1)

### Exercise 2 — top-k
Return the indices of the `k` largest entries of `s`, **in descending order of score**.

In [ ]:
def topk(s, k):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e2(fn):
    for _ in range(5):
        n = int(rng.integers(3, 10)); k = int(rng.integers(1, n + 1)); s = rng.normal(size=n)
        assert list(fn(s, k)) == list(np.argsort(-s)[:k])
    print("✅ Exercise 2 passed")

check_e2(topk)

### Exercise 3 — the retriever (capstone 🏁)
Put it together: given a query embedding `q`, a corpus `D` of shape `(n, d)`, and `k`, return the indices of the top-`k` documents by **cosine similarity**. The check plants a document the query is closest to and expects it ranked first.

In [ ]:
def retrieve(q, D, k):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e3(fn):
    d, n = 8, 6
    D = rng.normal(size=(n, d))
    q = 1.3 * D[4] + 0.01 * rng.normal(size=d)  # very close to document 4
    out = list(fn(q, D, 3))
    assert len(out) == 3 and len(set(out)) == 3
    assert out[0] == 4, f"top doc {out[0]} != 4"
    print("✅ Exercise 3 passed  —  you built a RAG retriever! 🎉")

check_e3(retrieve)

## Solutions (collapsed — peek only if stuck)

`ref_`-prefixed answers. Running the next cell validates everything against the checks above.

In [ ]:
def ref_build_matrix(vectors):
    return np.array(vectors)

def ref_normalize(x):
    return x / np.linalg.norm(x)

def ref_top1(s):
    return int(np.argmax(s))

def ref_topk(s, k):
    return np.argsort(-s)[:k]

def ref_retrieve(q, D, k):
    qn = q / np.linalg.norm(q)
    Dn = D / np.linalg.norm(D, axis=1, keepdims=True)
    sims = Dn @ qn
    return np.argsort(-sims)[:k]

check_r1(ref_build_matrix)
check_r2(ref_normalize)
check_e1(ref_top1)
check_e2(ref_topk)
check_e3(ref_retrieve)
print("\nAll reference solutions pass. 🎉  Milestone 1 complete — you can read & build RAG retrieval!")